# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [11]:
import math        # Para la función logaritmo (math.log)
import string      # Para obtener todos los signos de puntuación fácilmente
from collections import Counter  # Para contar palabras en un documento de forma rápida

In [2]:
ruta_archivo = "01_corpus_turismo_500.txt"

with open(ruta_archivo, 'r', encoding='utf-8') as archivo:
    # verificar líneas no vacías y eliminar espacios en blanco
    documentos_crudos = [linea.strip() for linea in archivo if linea.strip()]

print(f"Documentos leídos: {len(documentos_crudos)}")
print(f"Ejemplo — Documento 1: '{documentos_crudos[0]}'")

Documentos leídos: 500
Ejemplo — Documento 1: 'Otavalo es conocido por su mercado indígena y su artesanía Perfecto para rafting.'


In [3]:
documentos_tokenizados = []  # Lista de listas de palabras por documento
vocabulario_global = set()   # Conjunto de todas las palabras únicas del corpus

for texto_crudo in documentos_crudos:
    # a. convertir a minúsculas para que "Hotel" y "hotel" sean la misma palabra
    texto = texto_crudo.lower()

    # b. eliminar signos de puntuación 
    # string.punctuation contiene todos los signos de puntuación estándar
    for signo in string.punctuation:
        texto = texto.replace(signo, '')

    # c. separar en palabras individuales 
    palabras = texto.split()

    documentos_tokenizados.append(palabras)
    vocabulario_global.update(palabras)  # Agregar palabras nuevas al vocabulario

# Ordenar el vocabulario 
vocabulario_global = sorted(list(vocabulario_global))

cantidad_documentos = len(documentos_tokenizados)  # 'N' 

print(f"Total de documentos procesados: {cantidad_documentos}")
print(f"Tamaño del vocabulario: {len(vocabulario_global)} palabras únicas")
print(f"Primeras 10 palabras del vocabulario: {vocabulario_global[:10]}")


Total de documentos procesados: 500
Tamaño del vocabulario: 118 palabras únicas
Primeras 10 palabras del vocabulario: ['2000', 'a', 'agua', 'amazonía', 'arquitectura', 'artesanía', 'atrae', 'atraen', 'auténtico', 'aventura']


### Cálculo del IDF

Para cada palabra del vocabulario, contamos en cuántos documentos aparece (`df` = *document frequency*),
y luego aplicamos la fórmula:


$$idf_{t} = log( \frac{N}{df_{t}} )$$


**Nota:** Se utilizó `+ 1` en el denominador *(suavizado de Laplace)* para evitar `log(0)` en el
caso de que una palabra aparezca en absolutamente todos los documentos (df = N → log(1) = 0,
que no da error, pero con +1 el IDF mínimo sigue siendo > 0).

In [6]:
# idf es un diccionario: { palabra: valor_idf }

idf = {}

for palabra in vocabulario_global:
    # Contar en cuántos documentos aparece esta palabra (document frequency)
    df = sum(1 for doc in documentos_tokenizados if palabra in doc)

    # Fórmula IDF con suavizado +1 en el denominador para robustez
    idf[palabra] = math.log(cantidad_documentos / (df + 1))

print("IDF calculado para todas las palabras del vocabulario.")
print(f"Ejemplo — IDF de '{vocabulario_global[2]}': {idf[vocabulario_global[2]]:.4f}")


IDF calculado para todas las palabras del vocabulario.
Ejemplo — IDF de 'agua': 2.7181


### Cálculo de la matriz TF-IDF

Para cada documento calculamos el **TF** de cada palabra y multiplicamos por su **IDF**:

$$
tf_{t, d} = f_{t, d}
$$

$$
w_{t, d} = tf_{t, d}  \cdot idf_t
$$


El resultado es una lista de listas:
- **Filas** → documentos
- **Columnas** → palabras del vocabulario (en el mismo orden que `vocabulario_global`)

In [8]:
# matriz_tfidf[i][j] = valor TF-IDF de la palabra j en el documento i

matriz_tfidf = []

for doc in documentos_tokenizados:
    fila_del_documento = []
    total_palabras_en_doc = len(doc)

    # Counter crea un diccionario { palabra: cantidad_de_apariciones }
    conteo_palabras = Counter(doc)

    for palabra in vocabulario_global:
        # TF: si el documento no tiene palabras, TF = 0 
        if total_palabras_en_doc > 0:
            tf = conteo_palabras[palabra] / total_palabras_en_doc
        else:
            tf = 0

        # TF-IDF final
        valor_tfidf = tf * idf[palabra]
        fila_del_documento.append(round(valor_tfidf, 4))

    matriz_tfidf.append(fila_del_documento)

cantidad_filas    = len(matriz_tfidf)
cantidad_columnas = len(matriz_tfidf[0]) if matriz_tfidf else 0

print(f"Dimensión de la matriz TF-IDF: {cantidad_filas} filas x {cantidad_columnas} columnas")
print(f"  → {cantidad_filas} documentos, {cantidad_columnas} palabras en el vocabulario")
print()
print(f"Documento 1 original : '{documentos_crudos[0]}'")
print(f"TF-IDF del documento 1 (primeras 10 columnas): {matriz_tfidf[0][:100]}")


Dimensión de la matriz TF-IDF: 500 filas x 118 columnas
  → 500 documentos, 118 palabras en el vocabulario

Documento 1 original : 'Otavalo es conocido por su mercado indígena y su artesanía Perfecto para rafting.'
TF-IDF del documento 1 (primeras 10 columnas): [0.0, 0.0, 0.0, 0.0, 0.0, 0.2068, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2068, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1098, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2068, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2068, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.2068, 0.0334, 0.0, 0.0, 0.0, 0.123, 0.0, 0.0, 0.1085, 0.0, 0.0, 0.0, 0.1698, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.1696]


### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

En este paso, se transforman los archivos físicos (`.txt`) descargados del Proyecto Gutenberg en una estructura de datos en memoria que el algoritmo pueda procesar: Corpus. 

Consistirá en una lista de Python, donde cada elemento de la lista será el contenido completo de un libro. 

In [7]:
import os

ruta_gutenberg = "../data/gutenberg/1000/" 

# Verificar cantidad de archivos en el directorio
archivos = os.listdir(ruta_gutenberg)
print(f"Archivos encontrados en el directorio: {len(archivos)}")

Archivos encontrados en el directorio: 1000


In [8]:
# 'corpus_textos'   → lista con el texto crudo de cada libro
# 'nombres_libros'  → lista con el nombre del archivo (para saber qué libro es cada fila)

corpus_textos  = []
nombres_libros = []

for nombre_archivo in archivos:
    ruta_completa = os.path.join(ruta_gutenberg, nombre_archivo)

    with open(ruta_completa, "r", encoding="utf-8", errors="ignore") as archivo:
        texto = archivo.read()
        corpus_textos.append(texto)
        nombres_libros.append(nombre_archivo)

print(f"Corpus cargado: {len(corpus_textos)} libros en memoria.")
print(f"Ejemplo — primer libro: '{nombres_libros[0]}' ({len(corpus_textos[0])} caracteres)")


Corpus cargado: 1000 libros en memoria.
Ejemplo — primer libro: 'pg1.txt' (30086 caracteres)


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [9]:
# Librerías adicionales para el corpus grande
import math
from collections import Counter, defaultdict

In [16]:
libros_tokenizados = []  # Lista de listas de palabras (una sub-lista por libro)

print("Limpiando y tokenizando los 1000 libros...")

for nombre_archivo in archivos:
    ruta_completa = os.path.join(ruta_gutenberg, nombre_archivo)
    with open(ruta_completa, "r", encoding="utf-8", errors="ignore") as archivo:
        texto = archivo.read().lower()

        # Eliminar puntuación 
        for signo in string.punctuation:
            texto = texto.replace(signo, ' ')

        palabras = texto.split()
        libros_tokenizados.append(palabras)

cantidad_libros = len(libros_tokenizados)  # N en la fórmula IDF
print(f"Tokenización completada. Total de libros: {cantidad_libros}")


Limpiando y tokenizando los 1000 libros...
Tokenización completada. Total de libros: 1000


In [17]:
# Para cada palabra, saber en cuántos libros aparece.
# defaultdict(int) para no tener que inicializar cada clave en 0.

apariciones_en_libros = defaultdict(int)  # { palabra: cantidad_de_libros_donde_aparece }

print("Calculando document frequency (DF) de cada palabra...")

for palabras_del_libro in libros_tokenizados:
    # set() elimina duplicados: si "love" aparece 500 veces en un libro,
    # solo cuenta 1 para el DF
    palabras_unicas_del_libro = set(palabras_del_libro)
    for palabra in palabras_unicas_del_libro:
        apariciones_en_libros[palabra] += 1

# Calcular IDF con suavizado para evitar log(0)
idf_gutenberg = {}
for palabra, df in apariciones_en_libros.items():
    idf_gutenberg[palabra] = math.log(cantidad_libros / (df + 1))

print(f"IDF calculado. Vocabulario total: {len(idf_gutenberg):,} palabras únicas.")


Calculando document frequency (DF) de cada palabra...
IDF calculado. Vocabulario total: 1,212,222 palabras únicas.


In [ ]:
# La matriz se guarda como una lista de diccionarios.
# matriz_tfidf_gutenberg[i] = diccionario del libro i = { palabra: valor_tfidf }

matriz_tfidf_gutenberg = []

print("Calculando TF-IDF para cada libro...")

for palabras_del_libro in libros_tokenizados:
    tfidf_del_libro = {}  # Diccionario disperso, solo guarda palabras presentes

    total_palabras_en_libro = len(palabras_del_libro)

    # Counter cuenta cuántas veces aparece cada palabra en este libro
    conteo_palabras_libro = Counter(palabras_del_libro)

    for palabra, cantidad_apariciones in conteo_palabras_libro.items():
        # TF: frecuencia relativa de la palabra en este libro
        tf = cantidad_apariciones / total_palabras_en_libro

        # TF-IDF: multiplicamos TF por el IDF precalculado
        valor_tfidf = tf * idf_gutenberg[palabra]

        # Solo almacenamos si el valor es distinto de cero (IDF=0 → palabra en todos los libros)
        if valor_tfidf > 0:
            tfidf_del_libro[palabra] = round(valor_tfidf, 6)

    matriz_tfidf_gutenberg.append(tfidf_del_libro)

print("Matriz TF-IDF calculada.")


Calculando TF-IDF para cada libro...
Matriz TF-IDF calculada.


In [19]:
print("Dimensión lógica de la matriz TF-IDF Gutenberg:")
print(f"  Filas    (libros)     : {len(matriz_tfidf_gutenberg)}")
print(f"  Columnas (vocabulario): {len(idf_gutenberg):,} palabras únicas")
# print(matriz_tfidf_gutenberg) 

Dimensión lógica de la matriz TF-IDF Gutenberg:
  Filas    (libros)     : 1000
  Columnas (vocabulario): 1,212,222 palabras únicas


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [14]:
indice_invertido = {}  # { palabra: [archivo1, archivo2, ...] }

print("Construyendo índice invertido...")

for nombre_archivo in archivos:
    ruta_completa = os.path.join(ruta_gutenberg, nombre_archivo)
    with open(ruta_completa, "r", encoding="utf-8", errors="ignore") as archivo:
        texto = archivo.read().lower()

        # Limpiar puntuación
        for signo in string.punctuation:
            texto = texto.replace(signo, '')

        palabras_unicas_del_libro = set(texto.split())  # set → sin duplicados

        for palabra in palabras_unicas_del_libro:
            if palabra not in indice_invertido:
                indice_invertido[palabra] = []
            indice_invertido[palabra].append(nombre_archivo)

print(f"Índice construido. Palabras indexadas: {len(indice_invertido):,}")

Construyendo índice invertido...
Índice construido. Palabras indexadas: 2,100,346


In [12]:
# ── 2. Función de búsqueda AND ───────────────────────────────────────────────

def buscar(consulta, indice_invertido):
    """
    Busca los libros que contienen TODAS las palabras de la consulta (búsqueda AND).

    Parámetros
    ----------
    consulta : str
        Frase o palabras a buscar. Ejemplo: "great whale ocean"
    indice_invertido : dict
        Diccionario { palabra: [lista_de_archivos] } construido previamente.

    Retorna
    -------
    list
        Lista con los nombres de los archivos que contienen todas las palabras.
        Devuelve lista vacía si ningún libro cumple la condición.

    """
    # normalizar la consulta (minúsculas, separar en palabras)
    palabras_de_la_consulta = consulta.lower().split()

    if not palabras_de_la_consulta:
        return []  # Consulta vacía → sin resultados

    # empezar con los libros que tienen la primera palabra
    primera_palabra = palabras_de_la_consulta[0]
    libros_candidatos = set(indice_invertido.get(primera_palabra, []))
    # .get(clave, []) devuelve lista vacía si la palabra no está en el índice

    # intersección AND con el resto de palabras
    for palabra in palabras_de_la_consulta[1:]:
        libros_con_esta_palabra = set(indice_invertido.get(palabra, []))
        libros_candidatos = libros_candidatos & libros_con_esta_palabra

        # Optimización: si ya no quedan candidatos, salir 
        if not libros_candidatos:
            break

    return sorted(list(libros_candidatos))  # sorted para resultado reproducible


In [15]:
# Ejemplo de uso 

consulta_1 = "crimes"
resultados_1 = buscar(consulta_1, indice_invertido)
print(f"Resultados para '{consulta_1}': {len(resultados_1)} documentos encontrados.")

consulta_2 = "governor" 
resultados_2 = buscar(consulta_2, indice_invertido)
print(f"Resultados para '{consulta_2}': {len(resultados_2)} documentos encontrados.")

consulta_3 = "know"
resultados_3 = buscar(consulta_3, indice_invertido)
print(f"Resultados para '{consulta_3}': {len(resultados_3)} documentos encontrados.")

Resultados para 'crimes': 319 documentos encontrados.
Resultados para 'governor': 315 documentos encontrados.
Resultados para 'know': 997 documentos encontrados.
